In [1]:
import os
import json
import zipfile
import numpy as np
import copy
import requests

import pandas as pd
import geopandas as gpd

import shapely.wkt as wkt
from shapely.geometry import LineString
from shapely.geometry import Polygon
from shapely.geometry import Point

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import BoundaryNorm

from math import radians, cos, sin, asin, sqrt

In [ ]:
DIR="/home/dy23a.fsu/st/datasets/raw"
os.makedirs(DIR, exist_ok=True)
DIR=os.path.join(DIR,"Chicago")
os.makedirs(DIR, exist_ok=True)

Chicago_Data_Portal_KEY_ID = ""
Chicago_Data_Portal_KEY_SECRET = ""

chicago_shape="./Chicago Community Areas.geojson"
chi_shapefile_url="https://data.cityofchicago.org/api/v3/views/igwz-8jzy/query.geojson"

taxi_orig_file=os.path.join(DIR,"chi_taxi_2025_15min.csv")
TNP_orig_file=os.path.join(DIR,"chi_TNP_2025_15min.csv")
bike_orig_dir=os.path.join(DIR,"bike")
scooter_orig_file=os.path.join(DIR,"chi_scooter_2025_60min.csv")

# Download
Community Areas https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Boundaries-Community-Areas-Map/cauq-8yn6  
Taxi 2025 15min https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data    
Transportation Network Providers 2025 15min https://data.cityofchicago.org/Transportation/Transportation-Network-Providers-Trips-2025-/6dvr-xwnh/about_data  
Bike 2025 15min https://data.cityofchicago.org/Transportation/Divvy-Trips/fg6s-gzvg/about_data  
Scooter 2025 1hour https://data.cityofchicago.org/Transportation/E-Scooter-Trips/2i5w-ykuw/about_data  

## 1 Shapefile

In [3]:
if not os.path.exists(chicago_shape):
    response = requests.get(chi_shapefile_url)
    with open(chicago_shape, "w", encoding="utf-8") as file:
        file.write(response.text)

## 2 Taxi

In [4]:
import csv
import time
import random

from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except ImportError:
    from requests.packages.urllib3.util.retry import Retry

def _last_nonempty_line(path, tail_bytes=64 * 1024):
    with open(path, "rb") as fh:
        fh.seek(0, 2)
        size = fh.tell()
        if size == 0:
            return None
        fh.seek(max(0, size - tail_bytes))
        chunks = fh.read().split(b"\n")
        while chunks and not chunks[-1].strip():
            chunks.pop()
        return chunks[-1].decode("utf-8") if chunks else None

def _count_data_rows(path):
    with open(path, "rb") as fh:
        return max(0, sum(1 for _ in fh) - 1)

def _make_session():
    """Session with transport-level retry for DNS / connect failures."""
    session = requests.Session()
    session.auth = (Chicago_Data_Portal_KEY_ID, Chicago_Data_Portal_KEY_SECRET)
    retry = Retry(
        total=10,
        connect=10,
        read=0,            # read retries handled in outer loop with page-size shrink
        status=5,
        backoff_factor=2.0,        # 2, 4, 8, 16, ... seconds
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=32, pool_maxsize=32)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def download_from_api(API_URL, base_where, file, order_col, tie_col=None,
                      page_size=5000, max_retries=50, request_timeout=120,
                      min_page_size=2000):
    """Keyset-paginated downloader for Socrata SODA endpoints.

    Avoids deep $offset (which goes O(offset) on the server and times out beyond
    a few million rows). Resumes by reading the last row of `file` and using
    its (order_col, tie_col) as a cursor.

    On repeated ReadTimeouts the page size is halved (down to `min_page_size`)
    so a struggling endpoint can still make progress; it ramps back up after a
    successful page. DNS / connection failures use exponential backoff with
    jitter to ride out resolver flaps.
    """
    cursor_order = None
    cursor_tie = None
    header_cols = None
    header_written = False
    total = 0
    mode = "w"

    if os.path.exists(file) and os.path.getsize(file) > 0:
        with open(file, "r", encoding="utf-8") as fh:
            header_line = fh.readline().rstrip("\n")
        if header_line:
            header_cols = next(csv.reader([header_line]))
            last = _last_nonempty_line(file)
            if last and last != header_line:
                last_row = next(csv.reader([last]))
                row_dict = dict(zip(header_cols, last_row))
                cursor_order = row_dict.get(order_col)
                if tie_col:
                    cursor_tie = row_dict.get(tie_col)
                total = _count_data_rows(file)
                header_written = True
                mode = "a"
                tie_str = f", {tie_col}>'{cursor_tie}'" if cursor_tie else ""
                print(f"Resuming after {order_col}='{cursor_order}'{tie_str} "
                      f"({total} existing rows)")

    order_clause = f"{order_col} ASC" + (f", {tie_col} ASC" if tie_col else "")

    session = _make_session()

    cur_page_size = page_size

    with open(file, mode=mode, encoding="utf-8") as f:
        while True:
            if cursor_order is None:
                where = base_where
            elif tie_col and cursor_tie is not None:
                where = (
                    f"({base_where}) AND "
                    f"({order_col} > '{cursor_order}' OR "
                    f"({order_col} = '{cursor_order}' AND {tie_col} > '{cursor_tie}'))"
                )
            else:
                where = f"({base_where}) AND {order_col} > '{cursor_order}'"

            query = {"$where": where, "$order": order_clause, "$limit": cur_page_size}

            response = None
            for attempt in range(1, max_retries + 1):
                try:
                    response = session.get(API_URL, params=query, timeout=request_timeout)
                    break
                except requests.exceptions.RequestException as e:
                    is_timeout = isinstance(e, (requests.exceptions.ReadTimeout,
                                                requests.exceptions.ConnectTimeout))
                    is_conn = isinstance(e, requests.exceptions.ConnectionError)
                    msg = str(e)
                    is_dns = ("Name or service not known" in msg
                              or "Temporary failure in name resolution" in msg
                              or "nodename nor servname" in msg
                              or "getaddrinfo failed" in msg)

                    if is_timeout and cur_page_size > min_page_size:
                        cur_page_size = max(min_page_size, cur_page_size // 2)
                        query["$limit"] = cur_page_size
                        print(f"Attempt {attempt} timed out: {e}. "
                              f"Reducing page_size to {cur_page_size} ...")
                    if attempt == max_retries:
                        print(f"Failed after {max_retries} attempts: {e}")
                        raise

                    if is_dns or is_conn:
                        # DNS / connect failure: longer exponential backoff + jitter
                        # so concurrent threads don't retry in lockstep.
                        wait = min(300, (2 ** min(attempt, 8))) + random.uniform(0, 5)
                        kind = "DNS" if is_dns else "connection"
                        print(f"Attempt {attempt} {kind} error: {e}. "
                              f"Retrying in {wait:.1f}s ...")
                    else:
                        wait = min(120, 5 * attempt)
                        print(f"Attempt {attempt} failed: {e}. Retrying in {wait}s ...")
                    time.sleep(wait)

            if response.status_code != 200:
                print(f"Error: {response.status_code}")
                print(f"Info: {response.text}")
                return

            lines = response.text.splitlines()
            if len(lines) <= 1:
                break

            header_resp, data_lines = lines[0], lines[1:]
            if not header_written:
                f.write(header_resp + "\n")
                header_cols = next(csv.reader([header_resp]))
                header_written = True
            elif header_cols is None:
                header_cols = next(csv.reader([header_resp]))

            for line in data_lines:
                f.write(line + "\n")
            f.flush()

            last_row = next(csv.reader([data_lines[-1]]))
            row_dict = dict(zip(header_cols, last_row))
            cursor_order = row_dict.get(order_col)
            if tie_col:
                cursor_tie = row_dict.get(tie_col)

            n = len(data_lines)
            total += n
            print(f"Downloaded {total} (page_size={cur_page_size}) ...")

            if cur_page_size < page_size:
                cur_page_size = min(page_size, cur_page_size * 2)

            if n < query["$limit"]:
                break

    print(f"\nSaved {total}")

In [5]:
DOWNLOAD=False
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/resource/b8xg-w8bx.csv"
    base_where = (
        "trip_start_timestamp >= '2025-01-01T00:00:00.000' "
        "AND trip_start_timestamp < '2026-01-01T00:00:00.000'"
    )
    download_from_api(
        API_URL, base_where, taxi_orig_file,
        order_col="trip_start_timestamp", tie_col="trip_id",
    )

## 3 Scooter

In [6]:
DOWNLOAD=False
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/resource/2i5w-ykuw.csv"
    base_where = (
        "start_time >= '2025-01-01T00:00:00.000' "
        "AND start_time < '2026-01-01T00:00:00.000'"
    )
    download_from_api(
        API_URL, base_where, scooter_orig_file,
        order_col="start_time", tie_col="trip_id",
    )

## 4 Bike

In [7]:
DOWNLOAD=False
year = 2025
base_url_template = "https://divvy-tripdata.s3.amazonaws.com/{year}{month:02d}-divvy-tripdata.zip"
os.makedirs(bike_orig_dir, exist_ok=True)
if DOWNLOAD:
    for month in range(1, 13):
        url = base_url_template.format(year=year, month=month)
        zip_filename = f"{year}{month:02d}-divvy-tripdata.zip"
        zip_filepath = os.path.join(bike_orig_dir, zip_filename)
        try:
            response = requests.get(url, stream=True, timeout=120)

            if response.status_code == 200:
                with open(zip_filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192 * 1024):
                        if chunk:
                            f.write(chunk)

                with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
                    zip_ref.extractall(bike_orig_dir)
                os.remove(zip_filepath)

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")

## 5 TNP

In [8]:
from concurrent.futures import ThreadPoolExecutor, as_completed

DOWNLOAD=True
MAX_WORKERS=4
if DOWNLOAD:
    API_URL = "https://data.cityofchicago.org/resource/6dvr-xwnh.csv"
    year = 2025
    TNP_dir = os.path.join(DIR, "TNP")
    os.makedirs(TNP_dir, exist_ok=True)

    def _download_one_month(month):
        start = f"{year}-{month:02d}-01T00:00:00.000"
        if month == 12:
            end = f"{year + 1}-01-01T00:00:00.000"
        else:
            end = f"{year}-{month + 1:02d}-01T00:00:00.000"
        base_where = (
            f"trip_start_timestamp >= '{start}' "
            f"AND trip_start_timestamp < '{end}'"
        )
        month_file = os.path.join(TNP_dir, f"chi_TNP_{year}_{month:02d}_15min.csv")
        print(f"[{year}-{month:02d}] start -> {month_file}")
        download_from_api(
            API_URL, base_where, month_file,
            order_col="trip_start_timestamp", tie_col="trip_id",
        )
        print(f"[{year}-{month:02d}] done")
        return month

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(_download_one_month, m): m for m in range(1, 13)}
        for fut in as_completed(futures):
            m = futures[fut]
            try:
                fut.result()
            except Exception as e:
                print(f"[{year}-{m:02d}] failed: {e}")

[2025-01] start -> /home/dy23a.fsu/st/datasets/raw/Chicago/TNP/chi_TNP_2025_01_15min.csv
[2025-02] start -> /home/dy23a.fsu/st/datasets/raw/Chicago/TNP/chi_TNP_2025_02_15min.csv
[2025-03] start -> /home/dy23a.fsu/st/datasets/raw/Chicago/TNP/chi_TNP_2025_03_15min.csv
[2025-04] start -> /home/dy23a.fsu/st/datasets/raw/Chicago/TNP/chi_TNP_2025_04_15min.csv


Resuming after trip_start_timestamp='2025-02-28T23:45:00.000', trip_id>'ffdd86baddc99bd4d95084e9666c4e0aef9dc808' (7288575 existing rows)
Resuming after trip_start_timestamp='2025-04-30T23:45:00.000', trip_id>'fff1e05d363f494677a28762c86b0d7bcb251c80' (7535395 existing rows)
Resuming after trip_start_timestamp='2025-01-31T23:45:00.000', trip_id>'fff383ec88523143610ebd96347e64789c69c293' (7607290 existing rows)
Resuming after trip_start_timestamp='2025-03-31T23:45:00.000', trip_id>'ffa8db3c44de1d9c0f5b205aeb27a118e5c956d2' (8037030 existing rows)


Attempt 1 connection error: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Max retries exceeded with url: /resource/6dvr-xwnh.csv?%24where=%28trip_start_timestamp+%3E%3D+%272025-02-01T00%3A00%3A00.000%27+AND+trip_start_timestamp+%3C+%272025-03-01T00%3A00%3A00.000%27%29+AND+%28trip_start_timestamp+%3E+%272025-02-28T23%3A45%3A00.000%27+OR+%28trip_start_timestamp+%3D+%272025-02-28T23%3A45%3A00.000%27+AND+trip_id+%3E+%27ffdd86baddc99bd4d95084e9666c4e0aef9dc808%27%29%29&%24order=trip_start_timestamp+ASC%2C+trip_id+ASC&%24limit=5000 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)")). Retrying in 2.1s ...
Attempt 1 connection error: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Max retries exceeded with url: /resource/6dvr-xwnh.csv?%24where=%28trip_start_timestamp+%3E%3D+%272025-03-01T00%3A00%3A00.000%27+AND+trip_start_timestamp+%3C+%272025-04-01T00%3A00%3A00.000%27%29+AND+%28trip_star

Attempt 2 connection error: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Max retries exceeded with url: /resource/6dvr-xwnh.csv?%24where=%28trip_start_timestamp+%3E%3D+%272025-02-01T00%3A00%3A00.000%27+AND+trip_start_timestamp+%3C+%272025-03-01T00%3A00%3A00.000%27%29+AND+%28trip_start_timestamp+%3E+%272025-02-28T23%3A45%3A00.000%27+OR+%28trip_start_timestamp+%3D+%272025-02-28T23%3A45%3A00.000%27+AND+trip_id+%3E+%27ffdd86baddc99bd4d95084e9666c4e0aef9dc808%27%29%29&%24order=trip_start_timestamp+ASC%2C+trip_id+ASC&%24limit=5000 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)")). Retrying in 5.5s ...


Attempt 2 connection error: HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Max retries exceeded with url: /resource/6dvr-xwnh.csv?%24where=%28trip_start_timestamp+%3E%3D+%272025-03-01T00%3A00%3A00.000%27+AND+trip_start_timestamp+%3C+%272025-04-01T00%3A00%3A00.000%27%29+AND+%28trip_start_timestamp+%3E+%272025-03-31T23%3A45%3A00.000%27+OR+%28trip_start_timestamp+%3D+%272025-03-31T23%3A45%3A00.000%27+AND+trip_id+%3E+%27ffa8db3c44de1d9c0f5b205aeb27a118e5c956d2%27%29%29&%24order=trip_start_timestamp+ASC%2C+trip_id+ASC&%24limit=5000 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)")). Retrying in 4.3s ...


...

Downloaded 7610000 (page_size=5000) ...


Downloaded 7615000 (page_size=5000) ...


Downloaded 7620000 (page_size=5000) ...


Downloaded 7625000 (page_size=5000) ...


Downloaded 7627660 (page_size=5000) ...

Saved 7627660
[2025-11] done
